In [1]:
import sys
sys.path.append("..")   # so Python can find the src/ folder

from src.preprocess import load_data, separate_features_target, split_data, load_preprocessor

df = load_data("../data/raw/salary.csv")
X, y = separate_features_target(df)
X_train, X_test, y_train, y_test = split_data(X, y)

preprocessor = load_preprocessor("../models/preprocessor.pkl")

X_train_processed = preprocessor.transform(X_train)
X_test_processed = preprocessor.transform(X_test)

In [5]:
import pandas as pd
import numpy as np

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
import joblib

pd.set_option('display.max_columns', None)

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
print(X_train_processed.shape)   # (200000, 38)
print(X_test_processed.shape)    # (50000, 38)

(200000, 38)
(50000, 38)


In [6]:
# ==============================
# Step 2: Train the Linear Regression Model
# ==============================

model = LinearRegression()
model.fit(X_train_processed, y_train)

print("Model trained successfully.")
print("Intercept:", model.intercept_)
print("Number of coefficients:", len(model.coef_))

Model trained successfully.
Intercept: 105636.91051762058
Number of coefficients: 38


In [7]:
# ==============================
# Step 3: Evaluate the Model
# ==============================

# Predict on test set (unseen data)
y_pred = model.predict(X_test_processed)

# Calculate metrics
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("Mean Absolute Error (MAE):", round(mae, 2))
print("Root Mean Squared Error (RMSE):", round(rmse, 2))
print("R² Score:", round(r2, 4))

Mean Absolute Error (MAE): 6154.61
Root Mean Squared Error (RMSE): 7987.5
R² Score: 0.9541


In [8]:
print(df['salary'].describe())

count    250000.000000
mean     145718.080524
std       37407.952729
min       31867.000000
25%      119358.000000
50%      143453.000000
75%      169492.000000
max      333046.000000
Name: salary, dtype: float64


In [9]:
error_percentage = (6154.61 / 145718.08) * 100
print(f"MAE as % of average salary: {error_percentage:.2f}%")

MAE as % of average salary: 4.22%


In [10]:
# ==============================
# Step 4: Interpret Model Coefficients
# ==============================

# Get the exact output feature names from the preprocessor
feature_names = preprocessor.get_feature_names_out()

# Build a clean DataFrame pairing feature names with their coefficients
coefficients_df = pd.DataFrame({
    "feature": feature_names,
    "coefficient": model.coef_
})

# Sort by absolute impact (largest effect, positive or negative, first)
coefficients_df["abs_coefficient"] = coefficients_df["coefficient"].abs()
coefficients_df = coefficients_df.sort_values(by="abs_coefficient", ascending=False)

print(coefficients_df.drop(columns="abs_coefficient").to_string(index=False))

                                 feature   coefficient
                     nom__location_India -48161.744838
                       nom__location_USA  35687.366253
              nom__job_title_AI Engineer  27995.768504
             nom__job_title_Data Analyst -25750.534561
         nom__job_title_Business Analyst -23468.623473
                    nom__location_Canada  21666.168019
nom__job_title_Machine Learning Engineer  17352.756483
                   num__experience_years  16352.982088
                        nom__location_UK  14649.266026
       nom__job_title_Frontend Developer -12825.648889
          nom__job_title_Product Manager  11949.895162
                       ord__company_size  10608.829619
                    ord__education_level   8067.339578
                   nom__location_Germany   7687.906812
           nom__job_title_Cloud Engineer   6537.333741
                    nom__location_Sweden  -6349.575941
               nom__location_Netherlands  -6329.171106
        no

In [11]:
pd.set_option('display.max_rows', None)
print(coefficients_df.drop(columns="abs_coefficient").to_string(index=False))

                                 feature   coefficient
                     nom__location_India -48161.744838
                       nom__location_USA  35687.366253
              nom__job_title_AI Engineer  27995.768504
             nom__job_title_Data Analyst -25750.534561
         nom__job_title_Business Analyst -23468.623473
                    nom__location_Canada  21666.168019
nom__job_title_Machine Learning Engineer  17352.756483
                   num__experience_years  16352.982088
                        nom__location_UK  14649.266026
       nom__job_title_Frontend Developer -12825.648889
          nom__job_title_Product Manager  11949.895162
                       ord__company_size  10608.829619
                    ord__education_level   8067.339578
                   nom__location_Germany   7687.906812
           nom__job_title_Cloud Engineer   6537.333741
                    nom__location_Sweden  -6349.575941
               nom__location_Netherlands  -6329.171106
        no

In [12]:
coefficients_df.drop(columns="abs_coefficient").to_csv("../data/processed/coefficients.csv", index=False)
print("Saved to ../data/processed/coefficients.csv")

Saved to ../data/processed/coefficients.csv


In [13]:
# ==============================
# Step 5: Save the Trained Model
# ==============================

import os

os.makedirs("../models", exist_ok=True)

joblib.dump(model, "../models/linear_regression_model.pkl")

print("Model saved successfully to ../models/linear_regression_model.pkl")

Model saved successfully to ../models/linear_regression_model.pkl


In [14]:
import sys
sys.path.append("..")

from src.train_regression import run_training_pipeline

model, metrics = run_training_pipeline(
    data_path="../data/raw/salary.csv",
    preprocessor_path="../models/preprocessor.pkl",
    model_save_path="../models/linear_regression_model.pkl"
)

print(metrics)

{'mae': 6154.607845197999, 'rmse': np.float64(7987.504371946122), 'r2': 0.9540960533551986}
